<a href="https://colab.research.google.com/github/valentiafibri10/PraktikumEDA/blob/main/PROJECT_KELOMPOK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset_nilai_akademik_siswa.csv')
print(df.head()) # 5 baris pertama
print(df.info()) # tipe data & jumlah non-null tiap kolom
print(df.describe()) # statistik ringkas kolom numerik
print(df.shape) # jumlah (baris, kolom)

  id_siswa           nama     kelas   mata_pelajaran jenis_ujian  \
0  SIS0004  Joko Prasetyo  XI RPL 2              KKA         uas   
1  SIS0020      Eka Putri  XI RPL 1              PKK         uts   
2  SIS0015  Nanda Pratama  XI RPL 3  Pemrograman Web         UAS   
3  SIS0046  Fajar Nugroho  XI RPL 2       Matematika          UH   
4  SIS0011    Ayu Lestari  XI RPL 2  Pemrograman Web          UH   

     tanggal_ujian nilai guru_pengampu  
0       06/08/2026    46      Ibu Wati  
1  12 Agustus 2026  73,0  Bpk. Santoso  
2       2026-08-15    69  Bpk. Santoso  
3       10/08/2026    73   Bpk. Arifin  
4   6 Agustus 2026    53      Ibu Wati  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_siswa        79 non-null     object
 1   nama            79 non-null     object
 2   kelas           79 non-null     object
 3   mata_pelajaran  79

In [ ]:
print(df.isnull().sum()) # jumlah data kosong tiap kolom
print(df.duplicated().sum()) # jumlah baris duplikat
print(df['jenis_ujian'].unique()) # cek konsistensi kategori
print(df['tanggal_ujian'].unique()[:10]) # cek format tanggal

id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             4
guru_pengampu     4
dtype: int64
4
['uas' 'uts' 'UAS' 'UH' 'UTS' 'uh']
['06/08/2026' '12 Agustus 2026' '2026-08-15' '10/08/2026' '6 Agustus 2026'
 '08/08/2026' '19/08/2026' '11/08/2026' '2026-08-20' '3 Agustus 2026']


In [ ]:
# rapikan kolom nilai: hapus kata 'poin', ganti koma jadi titik, lalu ubah ke angka
df['nilai'] = df['nilai'].astype(str).str.replace(' poin', '', regex=False)
df['nilai'] = df['nilai'].str.replace(',', '.', regex=False)
df['nilai'] = pd.to_numeric(df['nilai'], errors='coerce')

# seragamkan kategori jenis_ujian jadi huruf kapital semua
df['jenis_ujian'] = df['jenis_ujian'].str.upper()

print(df['nilai'].describe())
print(df['jenis_ujian'].unique())

count     75.000000
mean      77.733333
std      108.908632
min       42.000000
25%       54.500000
50%       65.000000
75%       74.000000
max      999.000000
Name: nilai, dtype: float64
['UAS' 'UTS' 'UH']


In [ ]:
# nilai di luar rentang 0-100 dianggap salah input, dijadikan kosong (NaN) dulu
df.loc[(df['nilai'] < 0) | (df['nilai'] > 100), 'nilai'] = np.nan

# rapikan format tanggal_ujian yang campur 3 format jadi satu format datetime
bulan_map = {'januari':1,'februari':2,'maret':3,'april':4,'mei':5,'juni':6,
             'juli':7,'agustus':8,'september':9,'oktober':10,'november':11,'desember':12}

def rapikan_tanggal(x):
    x = str(x).strip()
    for nama, angka in bulan_map.items():
        if nama in x.lower():
            hari, _, tahun = x.split(' ')
            return pd.Timestamp(year=int(tahun), month=angka, day=int(hari))
    if '/' in x:
        hari, bulan, tahun = x.split('/')
        return pd.Timestamp(year=int(tahun), month=int(bulan), day=int(hari))
    return pd.to_datetime(x)  # sudah format YYYY-MM-DD

df['tanggal_ujian'] = df['tanggal_ujian'].apply(rapikan_tanggal)
print(df[['tanggal_ujian']].head())
print(df['tanggal_ujian'].dtype)

  tanggal_ujian
0    2026-08-06
1    2026-08-12
2    2026-08-15
3    2026-08-10
4    2026-08-06
datetime64[ns]


In [ ]:
df['guru_pengampu'] = df['guru_pengampu'].fillna('Tidak Diketahui') # isi kekosongan dengan label
df = df.dropna(subset=['nilai']) # hapus baris jika kolom nilai kosong
print(df.isnull().sum())

id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             0
guru_pengampu     0
dtype: int64


In [ ]:
print(df.duplicated().sum()) # jumlah baris duplikat
df = df.drop_duplicates()
df['nilai'] = df['nilai'].astype(float) # memastikan tipe data nilai adalah angka
print(df.dtypes)
print(df.shape)

4
id_siswa                  object
nama                      object
kelas                     object
mata_pelajaran            object
jenis_ujian               object
tanggal_ujian     datetime64[ns]
nilai                    float64
guru_pengampu             object
dtype: object
(70, 8)


In [ ]:
KKM = 75 # asumsi nilai minimal lulus, sesuaikan dengan KKM sekolah yang sebenarnya

tidak_lulus = df[df['nilai'] < KKM] # filtering
urut = df.sort_values(by='nilai', ascending=False) # sorting
df['status_kelulusan'] = np.where(df['nilai'] >= KKM, 'Lulus', 'Tidak Lulus') # kolom turunan
df['bulan_ujian'] = df['tanggal_ujian'].dt.strftime('%Y-%m') # kolom turunan
ringkasan = df.groupby('mata_pelajaran')['nilai'].mean().sort_values(ascending=False) # agregasi
print(tidak_lulus[['nama','mata_pelajaran','nilai']].head())
print(ringkasan)

            nama   mata_pelajaran  nilai
0  Joko Prasetyo              KKA   46.0
1      Eka Putri              PKK   73.0
2  Nanda Pratama  Pemrograman Web   69.0
3  Fajar Nugroho       Matematika   73.0
4    Ayu Lestari  Pemrograman Web   53.0
mata_pelajaran
Pemrograman Web    67.307692
Bahasa Inggris     67.166667
Matematika         66.090909
KKA                64.250000
PKK                63.312500
Basis Data         59.500000
Name: nilai, dtype: float64


In [ ]:
ringkasan_kelas = df.groupby('kelas')['nilai'].mean().sort_values(ascending=False)
ringkasan_ujian = df.groupby('jenis_ujian')['nilai'].mean()
persen_lulus = (df['status_kelulusan'] == 'Lulus').mean() * 100

print(ringkasan_kelas)
print(ringkasan_ujian)
print(f'Persentase lulus (KKM={KKM}): {persen_lulus:.1f}%')

kelas
XI RPL 1    67.235294
XI RPL 2    67.095238
XI RPL 3    60.968750
Name: nilai, dtype: float64
jenis_ujian
UAS    60.840000
UH     70.346154
UTS    60.684211
Name: nilai, dtype: float64
Persentase lulus (KKM=75): 20.0%


Nama Anggota Kelompok:
1. Purie Safanaya Prastanty(27)
2. Valentia Fibri Prastika(34)

Dataset yang dipilih:
Data Nilai Akademik Siswa

Pertanyaan Analisis Awal:
1. Mata pelajaran apa yang rata rata nilainya paling tinggi dan palig rendah
2. Kelas mana (XI RPL 1/2/3) yang performanya paling baik?
3. Berapa persentase siswa yang lulus KKM, dan siapa aja yang tidak lulus di lebih dari 1 mata pelajaran?

Dugaan Masalah Kualitas Data:
1. Missing value pada kolom: nilai (4 baris), guru_pengampu (4 baris)
2. Duplikat data (4 pasang baris identik)
3. Tipe data tidak sesuai pada kolom: nilai (masih teks, ada"xx poin" & koma desimal), tanggal_ujian (3 format berbeda)
4. Lainnya: kategori jenis_ujian tidak konsisten kapitalisasi (uas/UAS), aad outlier nilai=999

Rencana Teknik Pembersihan: Hapus baris duplikat-> konverssi nilai ke numerik (hapus"poin', ganti koma->titik, nilai di luar 0-100 dianggap tidak valid)-> parse tanggal_ujian ke format datetime standar-> seragamkan jenis_ujjian jadi huruf kapital -> isi guru_pengampu kosong dengan label "tidak diketahui" -> buang baris dengan nilai kosong/tidak valid

Rencana Manipulasi Data: Filter: siswa berstatus "Tidak Lulus"|Sort: nilai tertinggi->terendah|Kolom turunan: status_kelulusan, bulan_ujian|Groupby/agregasi:rata rata nilai per mata pelajaran, per kelas, per jenis ujian

Jadwal Kerja: P3 (Loading & Inspection): sudah selesai|P4 (Cleaning): sudah selesai|P5 (Manipulation): sudah selesai| P6 (Uji & Presentasi): menyusul-sesuaikan tanggal dengan jadwal kelasmu

Pembagian Peran:
1. Purie: pair programming semua tahap
2. Valentia: pair programming semua tahap